In [16]:
import os
import json 
import shutil
import nltk
from statistics import mean

In [17]:
def jaccard_similarity(output1, output2):
    set1 = set(output1)
    set2 = set(output2)
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union

def calculate_bleu(reference, hypothesis):
    reference = [reference]
    hypothesis = hypothesis
    reference_tokens = [nltk.word_tokenize(ref) for ref in reference]
    hypothesis_tokens = nltk.word_tokenize(hypothesis)
    # Berechne den BLEU-Score
    bleu_score = nltk.translate.bleu_score.sentence_bleu(reference_tokens, hypothesis_tokens)
    return bleu_score

In [42]:
def json_to_text(data):
    if isinstance(data, dict):
        if "AND" in data:
            left = json_to_text(data["AND"]["left"])
            right = json_to_text(data["AND"]["right"])
            return f"({left} AND {right})"
        elif "OR" in data:
            left = json_to_text(data["OR"]["left"])
            right = json_to_text(data["OR"]["right"])
            return f"({left} OR {right})"
        elif "NOT" in data:
            inner = json_to_text(data["NOT"]["left"])
            return f"(NOT {inner})"
        elif "raw_text" in data:
            return data["raw_text"]
    return ""
def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [43]:
model_name = "Llama-3-70B-Instruct_5_shot"
model_path = f"model_output/{model_name}/ready"
failed_inner_path = f"../model_output/{model_name}/failed_inner"
label_path = "../../chia_label/p2"
base_model_output_path = "model_output"

In [40]:
model_names = [d for d in os.listdir(base_model_output_path) if os.path.isdir(os.path.join(base_model_output_path, d))]

for model_name in model_names:
    model_path = os.path.join(base_model_output_path, model_name, "ready")
    failed_inner_path = os.path.join(base_model_output_path, model_name, "failed_inner")

    label_files = os.listdir(label_path)
    model_files = os.listdir(model_path)
    bleu_scores = []
    jaccard_similarities = []

    for label_file in label_files:
        base_name = label_file.split("_parsed_2.json")[0]

        # Extract model and nshot (taking into account that there might be two "temp" components)
        parts = model_name.split("_")
        model = "_".join(parts[:-2])
        nshot = parts[-2]
        model_file = f"{model}_{base_name}_{nshot}_shot.json"

        if model_file in model_files:
            label_file_path = os.path.join(label_path, label_file)
            model_file_path = os.path.join(model_path, model_file)

            label_data = read_json(label_file_path)
            #model_data = read_json(model_file_path)

            try:
                label_text = json_to_text(label_data)
                #model_text = json_to_text(model_data)

                #bleu_score = calculate_bleu(reference=label_text, hypothesis=model_text)
                #jaccard_score = jaccard_similarity(label_text, model_text)
                #bleu_scores.append(bleu_score)
                #jaccard_similarities.append(jaccard_score)

                #print(f"Label: {label_file}")
                #print(f"Model: {model_file}")
                #print(f"BLEU Score: {bleu_score}")
                #print(f"Jaccard Similarity: {jaccard_score}")

            except KeyError as e:
                print(f"Error processing file {label_file}: {e}")
                print(f"Error processing file {model_file}: {e}")
                failed_model_path = os.path.join(failed_inner_path, model_file)
                shutil.move(model_file_path, failed_model_path)

    # Durchschnittswerte berechnen
    if bleu_scores:
        average_bleu = mean(bleu_scores)
        average_jaccard = mean(jaccard_similarities)

        print(f"Durchschnittlicher BLEU Score für {model_name}: {average_bleu}")
        print(f"Durchschnittliche Jaccard Ähnlichkeit für {model_name}: {average_jaccard}")

In [31]:
# Durchschnittswerte berechnen
average_bleu = mean(bleu_scores)
average_jaccard = mean(jaccard_similarities)

print(f"Durchschnittlicher BLEU Score: {average_bleu}")
print(f"Durchschnittliche Jaccard Ähnlichkeit: {average_jaccard}")

In [30]:
# Label
label_text = json_to_text(label_data)
print(label_text)

In [31]:
# Model
model_text = json_to_text(model_data)
print(model_text)

In [33]:
calculate_bleu(reference=label_text , hypothesis=model_text)

In [1]:
def json_to_text(data):
    if isinstance(data, dict):
        if "AND" in data:
            left = json_to_text(data["AND"].get("left", {}))
            right = json_to_text(data["AND"].get("right", {}))
            return f"({left} AND {right})"
        elif "OR" in data:
            left = json_to_text(data["OR"].get("left", {}))
            right = json_to_text(data["OR"].get("right", {}))
            return f"({left} OR {right})"
        elif "NOT" in data:
            inner = json_to_text(data["NOT"].get("left", {}))
            return f"(NOT {inner})"
        elif "raw_text" in data:
            return data["raw_text"]
    return ""

In [2]:
data = {
    "AND":{
        "left":{
            "AND":{
                "left":{
                    "AND":{
                        "left":{
                            "AND":{
                                "left":{
                                    "raw_text":"Use of any investigational or non-registered product (drug or vaccine) other than the study vaccine(s) within 30 days preceding the first dose of study vaccine, or planned use during the study period"
                                }
                            }
                        },
                        "right":{
                            "OR":{
                                "left":{
                                    "raw_text":"Chronic administration (defined as more than 14 days) of immunosuppressants or other immune-modifying drugs within six months prior to the first vaccine dose"
                                }
                            }
                        }
                    }
                },
                "right":{
                    "AND":{
                        "left":{
                            "raw_text":"Planned administration/ administration of a vaccine not foreseen by the study protocol during the period starting one month before each dose of vaccine(s) and ending 7 days after dose 1 and dose 2 or 1 month after dose 3"
                        }
                    }
                }
            }
        },
        "right":{
            "AND":{
                "left":{
                    "OR":{
                        "left":{
                            "raw_text":"Previous vaccination against diphtheria, tetanus, pertussis, polio, hepatitis B, Haemophilus influenzae type b, and/or S. pneumoniae with the exception of vaccines where the first dose can be given within the first two weeks of life according to the national recommendations"
                        }
                    }
                }
            }
        }
    }
}

In [3]:
json_to_text(data)


'((((Use of any investigational or non-registered product (drug or vaccine) other than the study vaccine(s) within 30 days preceding the first dose of study vaccine, or planned use during the study period AND ) AND (Chronic administration (defined as more than 14 days) of immunosuppressants or other immune-modifying drugs within six months prior to the first vaccine dose OR )) AND (Planned administration/ administration of a vaccine not foreseen by the study protocol during the period starting one month before each dose of vaccine(s) and ending 7 days after dose 1 and dose 2 or 1 month after dose 3 AND )) AND ((Previous vaccination against diphtheria, tetanus, pertussis, polio, hepatitis B, Haemophilus influenzae type b, and/or S. pneumoniae with the exception of vaccines where the first dose can be given within the first two weeks of life according to the national recommendations OR ) AND ))'